# RFE용 데이터셋 제작
1. 고장 개체 비율 rfe_sample_train 8 : rfe_sample_test 2
2. serial_number 단위에서 failure 비율 유지하여 분할
3. 학습 세트는 정상 행은 배정받은 개체 내부에서 랜덤시드를 사용하여 고장 행의 10배수 샘플링
4. 테스트 세트는 정상 행은 배정받은 개체 내부에서 랜덤시드를 사용하여 고장 행의 100배수 샘플링
    - 원본 불균형 1 : 1405.8이지만 타협한 수치
- (같은 serial_number는 train과 test에 동시에 존재하면 안 됨)
- seed = 42

In [ ]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from collections import Counter

# ==========================================
# ⚙️ 실행 설정 (원하는 작업만 True로 바꾸세요)
# ==========================================
CONFIG = {
    "BUILD_TRAIN": True,  # 트레인 세트 제작 여부
    "BUILD_TEST": True,    # 테스트 세트 제작 여부
    "TRAIN_RATIO": 10,     # 트레인 정상 샘플링 배수 (1:10)
    "TEST_RATIO": 100,     # 테스트 정상 샘플링 배수 (1:100) - 메모리 최적화
}

# 1. 경로 설정
base_dir = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data'
target_file_name = 'rfe_sample_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)
output_train = os.path.join(base_dir, 'rfe_sample_train.parquet')
output_test = os.path.join(base_dir, 'rfe_sample_test.parquet')

# 2. 파일 필터링
all_files = [f for f in os.listdir(base_dir) if f.endswith('.parquet')]
feature_files = [
    f for f in all_files 
    if not f.startswith('tmp_') 
    and 'train' not in f.lower() 
    and 'test' not in f.lower() 
    and f != target_file_name
]

con = duckdb.connect()
# [메모리 가드] OOM 방지를 위한 안전 설정
con.execute("SET memory_limit = '20GB'")
con.execute("SET threads = 4")

# [Step 0] 청소 및 메타데이터 캐싱
print("🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...")
for f_name, build_flag in [('rfe_train.parquet', CONFIG["BUILD_TRAIN"]), ('rfe_test.parquet', CONFIG["BUILD_TEST"])]:
    if build_flag and f_name in os.listdir(base_dir):
        try: os.remove(os.path.join(base_dir, f_name))
        except: pass

file_meta = {}
for f in [target_file_name] + feature_files:
    f_path = os.path.join(base_dir, f)
    v_name = f"v_{f.replace('.', '_').replace('-', '_')}"
    con.execute(f"CREATE OR REPLACE VIEW {v_name} AS SELECT * FROM read_parquet('{f_path}')")
    cols = con.execute(f"SELECT * FROM {v_name} WHERE 1=0").df().columns.tolist()
    file_meta[f] = {"view": v_name, "columns": cols}

# [Step 1] SN 8:2 층화 분할
target_view = file_meta[target_file_name]["view"]
serial_stats = con.execute(f"SELECT serial_number, MAX(failure) as has_failed FROM {target_view} GROUP BY serial_number").df()
train_sn, test_sn = train_test_split(serial_stats['serial_number'], test_size=0.2, stratify=serial_stats['has_failed'], random_state=42)
con.register('train_sn_list', pd.DataFrame({'serial_number': train_sn}))
con.register('test_sn_list', pd.DataFrame({'serial_number': test_sn}))

def build_query(sn_table, output_file, ratio, seed):
    """지정된 비율로 샘플링하여 쿼리 빌드"""
    # 1. target_view(diff 파일)의 모든 컬럼 정보를 가져와 중복 방지 셋에 추가
    target_cols = file_meta[target_file_name]["columns"]
    seen_columns = set(target_cols) # serial_number, date, failure 포함 모든 차분 변수들
    
    select_parts = ["s.*"] # s.*은 이제 diff 파일의 모든 컬럼을 포함함
    join_parts = []
    
    for i, f in enumerate(feature_files):
        current_cols = file_meta[f]["columns"]
        v_name = file_meta[f]["view"]
        to_exclude = [col for col in current_cols if col in seen_columns]
        exclude_clause = ", ".join([f'"{c}"' for c in to_exclude])
        
        if to_exclude:
            select_parts.append(f'f{i}.* EXCLUDE ({exclude_clause})')
        else:
            select_parts.append(f"f{i}.*")
            
        join_parts.append(f"LEFT JOIN {v_name} f{i} USING (serial_number, date)")
        seen_columns.update(current_cols)

    fail_count = con.execute(f"SELECT COUNT(*) FROM {target_view} WHERE failure=1 AND serial_number IN (SELECT serial_number FROM {sn_table})").fetchone()[0]
    sample_rows = int(fail_count * ratio)
    
    main_sql = f"""
    WITH sampled_ids AS (
        -- [수정]: SELECT * 을 사용하여 diff 파일의 모든 변수(차분 등)를 유지
        SELECT * FROM {target_view} WHERE failure = 1 AND serial_number IN (SELECT serial_number FROM {sn_table})
        UNION ALL
        SELECT * FROM (
            SELECT * FROM {target_view}
            WHERE failure = 0 AND serial_number IN (SELECT serial_number FROM {sn_table})
        ) USING SAMPLE {sample_rows} ROWS (reservoir, {seed})
    )
    SELECT {", ".join(select_parts)} FROM sampled_ids s {" ".join(join_parts)}
    """
    return f"COPY ({main_sql}) TO '{output_file}' (FORMAT 'PARQUET')"


# [Step 2] 실행 (CONFIG 설정에 따라 분기)
if CONFIG["BUILD_TRAIN"]:
    print(f"🏗️ [Train] 제작 시작 (비율 1:{CONFIG['TRAIN_RATIO']})")
    con.execute(build_query("train_sn_list", output_train, CONFIG['TRAIN_RATIO'], seed=42))

if CONFIG["BUILD_TEST"]:
    print(f"🏗️ [Test] 제작 시작 (비율 1:{CONFIG['TEST_RATIO']})")
    con.execute(build_query("test_sn_list", output_test, CONFIG['TEST_RATIO'], seed=42))

# [검증]
print("\n⛪ [검증] 최종 데이터 분포 확인")
for name, path, run in [("TRAIN", output_train, CONFIG["BUILD_TRAIN"]), ("TEST", output_test, CONFIG["BUILD_TEST"])]:
    if run:
        stats = con.execute(f"SELECT SUM(CASE WHEN failure=1 THEN 1 ELSE 0 END) as fail, SUM(CASE WHEN failure=0 THEN 1 ELSE 0 END) as healthy FROM read_parquet('{path}')").df()
        print(f"[{name}] 고장: {stats['fail'][0]:,} / 정상: {stats['healthy'][0]:,} (비율 1:{round(stats['healthy'][0]/stats['fail'][0], 1)})")

con.close()
print("\n🚀 설정된 작업이 모두 완료되었습니다!")


🧹 충돌 방지를 위한 파일 청소 및 메타데이터 캐싱 중...
🏗️ [Train] 제작 시작 (비율 1:10)
🏗️ [Test] 제작 시작 (비율 1:100)

⛪ [검증] 최종 데이터 분포 확인
[TRAIN] 고장: 27,165.0 / 정상: 271,650.0 (비율 1:10.0)
[TEST] 고장: 6,819.0 / 정상: 681,900.0 (비율 1:100.0)

🚀 설정된 작업이 모두 완료되었습니다!


In [3]:
import duckdb
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# 1. 필수 경로 재설정
base_dir = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data'
target_file_name = 'rfe_sample_diff.parquet'
target_path = os.path.join(base_dir, target_file_name)

# 2. DuckDB 연결
con = duckdb.connect()

print("🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...")

# 3. 전체 시리얼 번호(SN) 목록과 고장 여부 가져오기
serial_stats = con.execute(f"""
    SELECT serial_number, MAX(failure) as has_failed 
    FROM read_parquet('{target_path}') 
    GROUP BY serial_number
""").df()

# 4. 8:2로 분할하여 '테스트 그룹' SN만 추출 (random_state 고정)
_, test_sn = train_test_split(
    serial_stats['serial_number'], 
    test_size=0.2, 
    stratify=serial_stats['has_failed'], 
    random_state=42
)

# 5. 테스트 SN 목록을 DuckDB에 임시 등록
con.register('test_sn_temp', pd.DataFrame({'serial_number': test_sn}))

# 6. 해당 그룹의 모든 행(전수조사)에 대한 불균형 비율 계산
result = con.execute(f"""
    SELECT 
        COUNT(CASE WHEN failure = 1 THEN 1 END) as fail_rows,
        COUNT(CASE WHEN failure = 0 THEN 1 END) as healthy_rows,
        ROUND(COUNT(CASE WHEN failure = 0 THEN 1 END) / NULLIF(COUNT(CASE WHEN failure = 1 THEN 1 END), 0), 2) as imbalance_ratio
    FROM read_parquet('{target_path}')
    WHERE serial_number IN (SELECT serial_number FROM test_sn_temp)
""").df()

print("\n📊 [테스트 SN 그룹(전수조사) 불균형 결과]")
print("-" * 40)
print(result.to_string(index=False))
print("-" * 40)
print(f"💡 결론: 테스트 개체들을 전수 조사하면 1:{result['imbalance_ratio'][0]} 비율이 나옵니다.")

con.close()


🔍 데이터 로드 및 8:2 분할 시뮬레이션 중...

📊 [테스트 SN 그룹(전수조사) 불균형 결과]
----------------------------------------
 fail_rows  healthy_rows  imbalance_ratio
      6779       9584672          1413.88
----------------------------------------
💡 결론: 테스트 개체들을 전수 조사하면 1:1413.88 비율이 나옵니다.


In [2]:
import duckdb
import os

# 1. 경로 설정
base_dir = r'C:\Workspace\06_ML_projdect\26_1_COIN\data\rfe_sample_data'
diff_path = os.path.join(base_dir, 'rfe_sample_diff.parquet')

# 원본 파일 및 결과 파일 설정
targets = [
    ('rfe_sample_train.parquet', 'rfe_sample_train.parquet'),
    ('rfe_sample_test.parquet', 'rfe_sample_test.parquet')
]

con = duckdb.connect()
# 메모리 및 스레드 최적화
con.execute("SET memory_limit = '20GB'")
con.execute("SET threads = 4")

print("⚡ 차분(diff) 변수 결합 작업을 시작합니다...")

for input_name, output_name in targets:
    input_path = os.path.join(base_dir, input_name)
    output_path = os.path.join(base_dir, output_name)
    
    # 쿼리 설명: 
    # main.* 은 기존 샘플링된 데이터의 모든 컬럼
    # diff.* EXCLUDE(...) 는 diff 파일에서 중복되는 키 컬럼을 제외한 모든 '차분 변수'들만 선택
    query = f"""
    COPY (
        SELECT 
            main.*, 
            diff.* EXCLUDE (serial_number, date, failure)
        FROM read_parquet('{input_path}') AS main
        LEFT JOIN read_parquet('{diff_path}') AS diff 
        USING (serial_number, date)
    ) TO '{output_path}' (FORMAT 'PARQUET')
    """
    
    print(f"🔄 {input_name} 처리 중...")
    con.execute(query)
    print(f"✅ {output_name} 저장 완료!")

con.close()
print("\n🚀 모든 파일에 차분 변수 결합이 완료되었습니다!")


⚡ 차분(diff) 변수 결합 작업을 시작합니다...
🔄 rfe_sample_train.parquet 처리 중...
✅ rfe_sample_train.parquet 저장 완료!
🔄 rfe_sample_test.parquet 처리 중...
✅ rfe_sample_test.parquet 저장 완료!

🚀 모든 파일에 차분 변수 결합이 완료되었습니다!
